# BirdCLEF 2026 — Final Submission Notebook

This notebook is aligned with experiment #78 / retrained exp78 family:
- input shape: `(64, 626, 3)`
- patched `.keras` full model loading
- official 5-second row IDs from `sample_submission.csv`
- placeholder submission only when hidden test audio is not mounted


In [1]:
import os, glob, json, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings("ignore")

print("TensorFlow:", tf.__version__)
print("Librosa:", librosa.__version__)


2026-05-06 11:24:28.216232: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778066668.511859      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778066668.600768      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778066669.271617      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778066669.271659      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778066669.271661      16 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
Librosa: 0.11.0


In [2]:
# =======================================================
# PATHS + EXP78 PARAMETERS
# =======================================================

ORIGINAL_MODEL = "/kaggle/input/datasets/danielemalerba0302/birdclef2026-model/exp78_full_model.keras"
TEST_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/birdclef-2026/sample_submission.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

SAMPLE_RATE = 32000
DURATION = 5.0

N_MELS = 64
N_FFT = 2048
HOP_LENGTH = 256
FMIN = 20
FMAX = 16000
TOP_DB = 40.0
MEL_NORM = "slaney"
USE_HTK = True

TARGET_HEIGHT = 64
TARGET_WIDTH = 626

TTA_OFFSETS = [-0.5, 0.0, 0.5]

print("Model exists:", os.path.exists(ORIGINAL_MODEL))
print("Test dir exists:", os.path.exists(TEST_AUDIO_DIR))
print("Sample submission exists:", os.path.exists(SAMPLE_SUB_PATH))

assert os.path.exists(ORIGINAL_MODEL), f"Missing model: {ORIGINAL_MODEL}"
assert os.path.exists(TEST_AUDIO_DIR), f"Missing test dir: {TEST_AUDIO_DIR}"
assert os.path.exists(SAMPLE_SUB_PATH), f"Missing sample submission: {SAMPLE_SUB_PATH}"


Model exists: True
Test dir exists: True
Sample submission exists: True


In [3]:
# =======================================================
# LOAD SAMPLE SUBMISSION / SPECIES ORDER
# =======================================================

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
SPECIES_LIST = list(sample_sub.columns[1:])

print("Sample submission shape:", sample_sub.shape)
print("Number of species:", len(SPECIES_LIST))
print(sample_sub.head())

assert len(SPECIES_LIST) == 234


Sample submission shape: (3, 235)
Number of species: 234
                                    row_id   1161364    116570   1176823  \
0   BC2026_Test_0001_S05_20250227_010002_5  0.004274  0.004274  0.004274   
1  BC2026_Test_0001_S05_20250227_010002_10  0.004274  0.004274  0.004274   
2  BC2026_Test_0001_S05_20250227_010002_15  0.004274  0.004274  0.004274   

    1491113   1595929    209233     22930     22956     22961  ...   whnjay1  \
0  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   
1  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   
2  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   

     whtdov   whwpic1    y00678    yebcar   yebela1    yecmac    yecpar  \
0  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   
1  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   
2  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   

    yehcar1   ye

In [4]:
# =======================================================
# PATCH .keras CONFIG FOR KAGGLE KERAS COMPATIBILITY
# =======================================================

PATCHED_MODEL = "/kaggle/working/exp78_full_model_patched.keras"
tmp_dir = "/kaggle/working/keras_patch_tmp"

if os.path.exists(tmp_dir):
    shutil.rmtree(tmp_dir)
os.makedirs(tmp_dir)

with zipfile.ZipFile(ORIGINAL_MODEL, "r") as z:
    z.extractall(tmp_dir)

config_path = os.path.join(tmp_dir, "config.json")

with open(config_path, "r") as f:
    config = json.load(f)

BAD_KEYS = [
    "renorm",
    "renorm_clipping",
    "renorm_momentum",
    "quantization_config",
]

def remove_bad_keys(obj):
    if isinstance(obj, dict):
        for key in BAD_KEYS:
            obj.pop(key, None)
        for value in obj.values():
            remove_bad_keys(value)
    elif isinstance(obj, list):
        for item in obj:
            remove_bad_keys(item)

remove_bad_keys(config)

with open(config_path, "w") as f:
    json.dump(config, f)

with zipfile.ZipFile(PATCHED_MODEL, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(tmp_dir):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, tmp_dir)
            z.write(full_path, arcname)

print("Patched model saved:", PATCHED_MODEL)


Patched model saved: /kaggle/working/exp78_full_model_patched.keras


In [5]:
# =======================================================
# LOAD PATCHED FULL MODEL
# =======================================================

model = keras.models.load_model(
    PATCHED_MODEL,
    compile=False,
    safe_mode=False
)

print("Model loaded successfully!")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

assert model.input_shape[1:] == (TARGET_HEIGHT, TARGET_WIDTH, 3)
assert model.output_shape[-1] == len(SPECIES_LIST)


2026-05-06 11:25:00.275682: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model loaded successfully!
Input shape: (None, 64, 626, 3)
Output shape: (None, 234)


In [6]:
# =======================================================
# PREPROCESSING — MUST MATCH EXP78 TRAINING PIPELINE
# =======================================================

def audio_to_spectrogram(audio_arr, sr=SAMPLE_RATE):
    mel = librosa.feature.melspectrogram(
        y=audio_arr,
        sr=sr,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        fmin=FMIN,
        fmax=FMAX,
        norm=MEL_NORM,
        htk=USE_HTK
    )

    mel = np.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)

    mel_db = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)
    mel_db = np.nan_to_num(mel_db, nan=-TOP_DB, posinf=0.0, neginf=-TOP_DB)

    mel_norm = (mel_db + TOP_DB) / TOP_DB
    mel_norm = np.clip(mel_norm, 0, 1)

    spec = np.stack([mel_norm, mel_norm, mel_norm], axis=-1).astype(np.float32)

    if spec.shape[1] < TARGET_WIDTH:
        spec = np.pad(
            spec,
            ((0, 0), (0, TARGET_WIDTH - spec.shape[1]), (0, 0)),
            mode="constant"
        )
    elif spec.shape[1] > TARGET_WIDTH:
        spec = spec[:, :TARGET_WIDTH, :]

    return spec


In [7]:
# =======================================================
# SHAPE SANITY CHECK
# =======================================================

dummy = np.zeros(int(SAMPLE_RATE * DURATION), dtype=np.float32)
test_spec = audio_to_spectrogram(dummy)

print("Test spectrogram shape:", test_spec.shape)
print("Model expected shape:", model.input_shape[1:])

assert test_spec.shape == model.input_shape[1:]


Test spectrogram shape: (64, 626, 3)
Model expected shape: (64, 626, 3)


In [8]:
# =======================================================
# PREDICT EXACT ROW_IDS REQUESTED BY SAMPLE_SUBMISSION
# 3-view shifted TTA: average raw predictions per row_id,
# then apply sqrt calibration once after averaging.
# =======================================================

def extract_shifted_block(y, start_second, offset_second):
    block_samples = int(DURATION * SAMPLE_RATE)

    start_sample = int(round((start_second + offset_second) * SAMPLE_RATE))
    end_sample = start_sample + block_samples

    left_pad = max(0, -start_sample)
    right_pad = max(0, end_sample - len(y))

    start_sample = max(0, start_sample)
    end_sample = min(len(y), end_sample)

    block = y[start_sample:end_sample]

    if left_pad or right_pad:
        block = np.pad(block, (left_pad, right_pad), mode="constant")

    if len(block) < block_samples:
        block = np.pad(block, (0, block_samples - len(block)), mode="constant")
    elif len(block) > block_samples:
        block = block[:block_samples]

    return block


def predict_file_from_expected_rows(audio_path, expected_rows_for_file):
    y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

    results = []

    for row_id in expected_rows_for_file:
        end_second = int(row_id.split("_")[-1])
        start_second = end_second - int(DURATION)

        for offset in TTA_OFFSETS:
            block = extract_shifted_block(y, start_second, offset)
            spec = audio_to_spectrogram(block, sr=sr)
            results.append((row_id, spec))

    return results


In [ ]:
# =======================================================
# INFERENCE
# In visible runs, test_soundscapes contains only readme.txt.
# In the hidden rerun, Kaggle should populate the real .ogg files.
# =======================================================

audio_files = []

for ext in ["*.ogg", "*.wav", "*.flac", "*.mp3"]:
    audio_files.extend(
        glob.glob(os.path.join(TEST_AUDIO_DIR, "**", ext), recursive=True)
    )

audio_files = sorted(audio_files)

print("TEST_AUDIO_DIR:", TEST_AUDIO_DIR)
print("Found test audio files:", len(audio_files))
print("First files:", audio_files[:5])
print("TTA offsets:", TTA_OFFSETS)

expected_by_file = {}
for row_id in sample_sub["row_id"]:
    file_stem = "_".join(row_id.split("_")[:-1])
    expected_by_file.setdefault(file_stem, []).append(row_id)

print("Expected files:", len(expected_by_file))
print("First expected files:", list(expected_by_file.keys())[:3])

predictions_by_row_id = {}
BATCH_SIZE = 32
batch_specs = []
batch_row_ids = []

def flush_batch():
    global batch_specs, batch_row_ids

    if len(batch_specs) == 0:
        return

    X = np.array(batch_specs, dtype=np.float32)
    preds = model.predict(X, verbose=0)
    preds = np.clip(preds, 0.0, 1.0)

    for row_id, pred in zip(batch_row_ids, preds):
        predictions_by_row_id.setdefault(row_id, []).append(pred)

    batch_specs = []
    batch_row_ids = []

for i, audio_path in enumerate(audio_files):
    file_stem = os.path.splitext(os.path.basename(audio_path))[0]

    if i % 10 == 0:
        print(f"{i+1}/{len(audio_files)}: {file_stem}")

    if file_stem not in expected_by_file:
        print("Skipping unexpected file:", file_stem)
        continue

    blocks = predict_file_from_expected_rows(audio_path, expected_by_file[file_stem])

    for row_id, spec in blocks:
        batch_row_ids.append(row_id)
        batch_specs.append(spec)

        if len(batch_specs) >= BATCH_SIZE:
            flush_batch()

flush_batch()

for row_id in list(predictions_by_row_id.keys()):
    raw_avg = np.mean(predictions_by_row_id[row_id], axis=0)
    predictions_by_row_id[row_id] = np.sqrt(np.clip(raw_avg, 0.0, 1.0))

print("Predicted row_ids:", len(predictions_by_row_id))


TEST_AUDIO_DIR: /kaggle/input/competitions/birdclef-2026/test_soundscapes
Found test audio files: 0
First files: []
Expected files: 1
First expected files: ['BC2026_Test_0001_S05_20250227_010002']
Predicted row_ids: 0


In [10]:
# =======================================================
# BUILD SUBMISSION
# If no test audio is mounted, create a valid placeholder for dummy visible run.
# In hidden rerun, predictions_by_row_id should be populated and real predictions are saved.
# =======================================================

expected_row_ids = list(sample_sub["row_id"])

if len(predictions_by_row_id) == 0:
    print("No predictions generated in this environment.")
    print("Creating placeholder submission so the notebook can commit.")

    placeholder = np.zeros(
        (len(expected_row_ids), len(SPECIES_LIST)),
        dtype=np.float32
    )

    submission_df = pd.DataFrame(placeholder, columns=SPECIES_LIST)
    submission_df.insert(0, "row_id", expected_row_ids)

else:
    missing = sorted(set(expected_row_ids) - set(predictions_by_row_id.keys()))
    extra = sorted(set(predictions_by_row_id.keys()) - set(expected_row_ids))

    print("Missing rows:", len(missing))
    print("Extra rows:", len(extra))

    assert not missing, f"Missing row_ids: {missing[:10]}"
    assert not extra, f"Extra row_ids: {extra[:10]}"

    predictions_array = np.array(
        [predictions_by_row_id[row_id] for row_id in expected_row_ids],
        dtype=np.float32
    )

    submission_df = pd.DataFrame(predictions_array, columns=SPECIES_LIST)
    submission_df.insert(0, "row_id", expected_row_ids)

submission_df.to_csv(SUBMISSION_PATH, index=False)

print("Saved:", SUBMISSION_PATH)
print("Submission shape:", submission_df.shape)
print(submission_df.head())


No predictions generated in this environment.
Creating placeholder submission so the notebook can commit.
Saved: /kaggle/working/submission.csv
Submission shape: (3, 235)
                                    row_id  1161364  116570  1176823  1491113  \
0   BC2026_Test_0001_S05_20250227_010002_5      0.0     0.0      0.0      0.0   
1  BC2026_Test_0001_S05_20250227_010002_10      0.0     0.0      0.0      0.0   
2  BC2026_Test_0001_S05_20250227_010002_15      0.0     0.0      0.0      0.0   

   1595929  209233  22930  22956  22961  ...  whnjay1  whtdov  whwpic1  \
0      0.0     0.0    0.0    0.0    0.0  ...      0.0     0.0      0.0   
1      0.0     0.0    0.0    0.0    0.0  ...      0.0     0.0      0.0   
2      0.0     0.0    0.0    0.0    0.0  ...      0.0     0.0      0.0   

   y00678  yebcar  yebela1  yecmac  yecpar  yehcar1  yeofly1  
0     0.0     0.0      0.0     0.0     0.0      0.0      0.0  
1     0.0     0.0      0.0     0.0     0.0      0.0      0.0  
2     0.0     0.0 

In [11]:
# =======================================================
# VERIFICATION
# =======================================================

check_df = pd.read_csv(SUBMISSION_PATH)
sample_check = pd.read_csv(SAMPLE_SUB_PATH)

print("=== VERIFICATION ===")
print("Rows:", len(check_df))
print("Columns:", len(check_df.columns), "expected", len(sample_check.columns))

columns_match = list(check_df.columns) == list(sample_check.columns)
print("Columns match:", columns_match)
assert columns_match

row_ids_match = list(check_df["row_id"]) == list(sample_check["row_id"])
print("Row IDs match:", row_ids_match)
assert row_ids_match

pred_values = check_df.iloc[:, 1:].values

print("Min:", pred_values.min())
print("Max:", pred_values.max())
print("NaN:", np.isnan(pred_values).any())

assert not np.isnan(pred_values).any()
assert pred_values.min() >= 0.0
assert pred_values.max() <= 1.0

print("Submission ready:", SUBMISSION_PATH)


=== VERIFICATION ===
Rows: 3
Columns: 235 expected 235
Columns match: True
Row IDs match: True
Min: 0.0
Max: 0.0
NaN: False
Submission ready: /kaggle/working/submission.csv
